# 00 - Déduplication et split propre (Mendeley BT-MRI)

Je déduplique le dataset Mendeley BT-MRI par hachage perceptuel, refais un split 80/20 sans fuite au niveau des groupes de quasi-doublons et copie le résultat dans le dossier clean.

### Imports & configuration

In [1]:
import os, glob, re, shutil, random
import numpy as np
import imagehash
from PIL import Image
from collections import defaultdict

SEED       = 42
random.seed(SEED); np.random.seed(SEED)

SRC        = "/Users/teodul/Documents/Memoire/Brain Tumor MRI Dataset"  # dataset brut (train/ test/)
DST        = "/Users/teodul/Documents/Memoire/Brain Tumor MRI clean" # sortie propre
CANON      = ["glioma", "meningioma", "notumor", "pituitary"]
DUP_T      = 5      # seuil Hamming pour considérer deux images comme quasi-doublons
TEST_FRAC  = 0.20   # proportion test
HASH_SIZE  = 8      # pHash 64 bits

def phash_int(p):
    return int(str(imagehash.phash(Image.open(p).convert("L"), hash_size=HASH_SIZE)), 16)

def bits(arr):
    return np.unpackbits(arr.view(np.uint8).reshape(-1, 8), axis=1)

def canon_class(folder):
    n = folder.lower().replace("_", "").replace(" ", "").replace("-", "")
    if "no" in n and "tumor" in n: return "notumor"
    if n.startswith("gli"):        return "glioma"
    if n.startswith("men"):        return "meningioma"
    if n.startswith("pit"):        return "pituitary"
    return None

def get_patient_id(path):
    f = os.path.basename(path)
    for pat in [r"^(?:Tr|Te)-([a-z]{2}_\d+)", r"^([A-Z]_\d+)_", r"^([A-Z]_\d+)\.", r"^(\d+)"]:
        m = re.match(pat, f)
        if m: return m.group(1)
    return None

### 1. Collecte de toutes les images (train + test)

In [2]:
def find_split_dir(root, name):
    for d in os.listdir(root):
        if d.lower() == name and os.path.isdir(os.path.join(root, d)):
            return os.path.join(root, d)
    return None

train_dir = find_split_dir(SRC, "train")
test_dir  = find_split_dir(SRC, "test")
print("train:", train_dir)
print("test :", test_dir)

items = []  # (path, classe_canonique)
for sd in [train_dir, test_dir]:
    if sd is None:
        continue
    for folder in sorted(os.listdir(sd)):
        cpath = os.path.join(sd, folder)
        if not os.path.isdir(cpath):
            continue
        cc = canon_class(folder)
        if cc is None:
            print("  /!\\ dossier ignoré (classe inconnue):", folder)
            continue
        for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
            for p in sorted(glob.glob(os.path.join(cpath, ext))):
                items.append((p, cc))

print(f"\nimages totales : {len(items)}")
for cc in CANON:
    print(f"  {cc:11s}: {sum(1 for _, c in items if c == cc)}")

train: /Users/teodul/Documents/Memoire/Brain Tumor MRI Dataset/Train
test : /Users/teodul/Documents/Memoire/Brain Tumor MRI Dataset/Test

images totales : 12064
  glioma     : 3773
  meningioma : 2729
  notumor    : 2432
  pituitary  : 3130


### 2. Hachage perceptuel (quelques secondes)

In [3]:
hashes = np.array([phash_int(p) for p, _ in items], dtype=np.uint64)

# regroupement par hash exact (collapse rapide des doublons identiques)
hash_to_items = defaultdict(list)
for i, h in enumerate(hashes):
    hash_to_items[int(h)].append(i)

uniq_hashes = np.array(sorted(hash_to_items.keys()), dtype=np.uint64)
print(f"hashs uniques : {len(uniq_hashes)} / {len(items)} images")

hashs uniques : 8148 / 12064 images


### 3. Déduplication - un représentant par cluster de quasi-doublons

In [4]:
UB = bits(uniq_hashes)
parent = list(range(len(uniq_hashes)))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[ra] = rb

# union des hashs uniques distants de <= DUP_T  (peut prendre ~1-2 min)
for i in range(len(uniq_hashes)):
    d = (UB ^ UB[i]).sum(axis=1)
    for j in np.where(d <= DUP_T)[0]:
        if j > i:
            union(i, int(j))

clusters = defaultdict(list)
for hi in range(len(uniq_hashes)):
    clusters[find(hi)].append(hi)

reps = []  # un index d'image par cluster
for his in clusters.values():
    member_items = []
    for hi in his:
        member_items += hash_to_items[int(uniq_hashes[hi])]
    reps.append(sorted(member_items)[0])

print(f"images uniques après dédup : {len(reps)} / {len(items)}")

images uniques après dédup : 5755 / 12064


### 4. Split 80/20 au niveau patient/cluster (anti-fuite)

In [5]:
groups = defaultdict(list)
for i in reps:
    p, cc = items[i]
    pid = get_patient_id(p)
    key = (cc, pid) if pid else (cc, f"solo_{i}")
    groups[key].append(i)

by_class = defaultdict(list)
for key, mem in groups.items():
    by_class[key[0]].append(mem)

rng = random.Random(SEED)
train_idx, test_idx = [], []
for cc in CANON:
    gs = by_class.get(cc, [])
    rng.shuffle(gs)
    n_total = sum(len(m) for m in gs)
    n_test  = max(1, int(TEST_FRAC * n_total))
    cnt = 0
    for g in gs:
        if cnt < n_test:
            test_idx += g; cnt += len(g)
        else:
            train_idx += g

print("Split :")
for split, idxs in [("Train", train_idx), ("Test", test_idx)]:
    per = {cc: sum(1 for i in idxs if items[i][1] == cc) for cc in CANON}
    print(f"  {split:5s} total={len(idxs):5d}  {per}")

Split :
  Train total= 4606  {'glioma': 1687, 'meningioma': 1148, 'notumor': 592, 'pituitary': 1179}
  Test  total= 1149  {'glioma': 421, 'meningioma': 286, 'notumor': 147, 'pituitary': 295}


### 5. Copie vers le dossier clean

In [6]:
assert "clean" in DST.lower(), "Sécurité : DST doit contenir 'clean'."
if os.path.exists(DST):
    shutil.rmtree(DST)

def copy_set(idxs, split):
    for i in idxs:
        p, cc = items[i]
        out = os.path.join(DST, split, cc)
        os.makedirs(out, exist_ok=True)
        shutil.copy2(p, os.path.join(out, f"{i}_{os.path.basename(p)}"))

copy_set(train_idx, "Train")
copy_set(test_idx,  "Test")
print("Copié vers", DST)

Copié vers /Users/teodul/Documents/Memoire/Brain Tumor MRI clean


### 6. Test d'acceptation

In [8]:
def hashes_of(paths):
    return np.array([phash_int(p) for p in paths], dtype=np.uint64)

trp = glob.glob(os.path.join(DST, "Train", "**", "*.*"), recursive=True)
tep = glob.glob(os.path.join(DST, "Test",  "**", "*.*"), recursive=True)
HTR = bits(hashes_of(trp))
HTE = bits(hashes_of(tep))

mind = np.array([(HTR ^ HTE[i]).sum(axis=1).min() for i in range(len(HTE))])
print(f"test = {len(mind)} images")
for thr in [0, 2, 4, DUP_T]:
    print(f"  test ayant un doublon dans le train <= {thr} : {(mind <= thr).sum()}")
print("\n", "OK - aucun doublon" if (mind <= DUP_T).sum() == 0 else "ATTENTION - fuite résiduelle, augmente DUP_T")

test = 1149 images
  test ayant un doublon dans le train <= 0 : 0
  test ayant un doublon dans le train <= 2 : 0
  test ayant un doublon dans le train <= 4 : 0
  test ayant un doublon dans le train <= 5 : 0

 OK - aucun doublon
